# Day 6: Advanced Analytics & Risk Metrics

## Objectives
- Implement **Value at Risk (VaR)** and **Conditional Value at Risk (CVaR)** to assess downside risk.
- Compute **Rolling 90-day Sharpe Ratios** to track risk-adjusted performance volatility.
- Conduct **Investor Cohort Analysis** by first transaction year (2024 vs 2025).
- Analyze **SIP Continuation** and identify churn risk (gaps > 35 days).
- Assess **Sector Concentration Risk** using the **Herfindahl-Hirschman Index (HHI)** for equity portfolios.
- Document key findings and insights.

### Mathematical Rationale & Formulas

1. **Historical Value at Risk (VaR) at 95% Confidence**:
   $$\text{VaR}_{95\%} = -P_{5}(R_t)$$
   where $P_{5}$ is the 5th percentile of the daily returns distribution $R_t$. It represents the maximum expected loss with 95% confidence over a 1-day horizon.

2. **Conditional Value at Risk (CVaR) at 95% Confidence**:
   $$\text{CVaR}_{95\%} = -E[R_t \mid R_t \le -\text{VaR}_{95\%}]$$
   CVaR calculates the expected loss in the worst 5% of cases (the tail risk).

3. **Rolling 90-day Sharpe Ratio**:
   $$\text{Rolling Sharpe}_t = \frac{\mu_{90, t}}{\sigma_{90, t}} \times \sqrt{252}$$
   where $\mu_{90, t}$ and $\sigma_{90, t}$ are the mean and standard deviation of daily returns over the trailing 90 days.

4. **Sector Herfindahl-Hirschman Index (HHI)**:
   $$\text{HHI} = \sum_{i=1}^{N} w_i^2$$
   where $w_i$ is the percentage weight of sector $i$ in the fund's equity portfolio. 
   - $\text{HHI} > 2500$: Highly Concentrated portfolio risk.
   - $1500 \le \text{HHI} \le 2500$: Moderately Concentrated portfolio risk.
   - $\text{HHI} < 1500$: Diverse (low sector concentration).

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Setup plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams["figure.figsize"] = (12, 6)

BASE_DIR = Path("..")
DB_PATH = BASE_DIR / "database" / "bluestock_mf.db"
REPORTS_DIR = BASE_DIR / "reports" / "day6"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Connect to the database
conn = sqlite3.connect(DB_PATH)

# Load tables
dim_fund = pd.read_sql_query("SELECT * FROM dim_fund", conn)
fact_nav = pd.read_sql_query("SELECT * FROM fact_nav", conn)
fact_transactions = pd.read_sql_query("SELECT * FROM fact_transactions", conn)
fact_holdings = pd.read_sql_query("SELECT * FROM fact_holdings", conn)

conn.close()
print(f"Loaded {len(dim_fund)} funds, {len(fact_nav)} NAV records, {len(fact_transactions)} transactions, and {len(fact_holdings)} holdings.")

## 1. Value at Risk (VaR) & Conditional VaR (CVaR)

We calculate the 95% historical Value at Risk (VaR) and CVaR for each fund. These metrics highlight the potential downside risk of investing in each fund.

In [ ]:
var_cvar_results = []
for code, group in fact_nav.groupby("amfi_code"):
    returns = group["daily_return"].dropna()
    if len(returns) == 0:
        continue
    
    var_95 = returns.quantile(0.05)
    cvar_95 = returns[returns <= var_95].mean()
    scheme_name = dim_fund[dim_fund["amfi_code"] == code]["scheme_name"].values[0]
    
    var_cvar_results.append({
        "amfi_code": code,
        "scheme_name": scheme_name,
        "var_95_pct": -var_95 * 100,
        "cvar_95_pct": -cvar_95 * 100
    })

var_cvar_df = pd.DataFrame(var_cvar_results).sort_values("var_95_pct", ascending=False).reset_index(drop=True)
print("Top 5 Funds with Highest Downside Risk (95% VaR):")
print(var_cvar_df.head())

print("\nTop 5 Funds with Lowest Downside Risk (95% VaR):")
print(var_cvar_df.tail())

# Save CSV
var_cvar_df.to_csv(REPORTS_DIR / "var_cvar_report.csv", index=False)

## 2. Rolling 90-day Sharpe Ratio

Rolling Sharpe ratios allow us to see how a fund's risk-adjusted performance evolves. We evaluate 5 major funds (Mirae Asset Large Cap, ICICI Pru Midcap, Kotak Flexicap, HDFC Mid-Cap, and ICICI Pru Bluechip) over time.

In [ ]:
nav_pivot = fact_nav.pivot(index="nav_date", columns="amfi_code", values="daily_return").sort_index()
selected_codes = [148567, 120505, 120843, 100033, 120504]
selected_codes = [c for c in selected_codes if c in nav_pivot.columns]

fig, ax = plt.subplots(figsize=(14, 7), dpi=150)
fig.patch.set_facecolor('#fafafa')
ax.set_facecolor('#ffffff')

COLORS = ["#0F766E", "#2563EB", "#7C3AED", "#DC2626", "#F59E0B"]

for idx, code in enumerate(selected_codes):
    fund_returns = nav_pivot[code].dropna()
    rolling_mean = fund_returns.rolling(90).mean()
    rolling_std = fund_returns.rolling(90).std()
    rolling_sharpe = (rolling_mean / rolling_std) * np.sqrt(252)
    
    name = dim_fund[dim_fund["amfi_code"] == code]["scheme_name"].values[0].split(" - ")[0]
    rolling_sharpe = rolling_sharpe.dropna()
    dates = pd.to_datetime(rolling_sharpe.index)
    ax.plot(dates, rolling_sharpe, label=name, color=COLORS[idx % len(COLORS)], linewidth=2)

ax.set_title("Rolling 90-day Sharpe Ratio (Trailing Risk-Adjusted Return)", fontsize=14, fontweight='bold', pad=15, color='#1e293b')
ax.set_xlabel("Date", fontsize=11, color='#475569', labelpad=10)
ax.set_ylabel("Sharpe Ratio (Annualized)", fontsize=11, color='#475569', labelpad=10)
ax.grid(True, linestyle="--", alpha=0.5, color="#cbd5e1")
ax.legend(loc="upper left", frameon=True, facecolor="white", edgecolor="#e2e8f0")

plt.savefig(REPORTS_DIR / "rolling_sharpe_chart.png", bbox_inches="tight", facecolor='#fafafa')
plt.show()

## 3. Investor Cohort Analysis

We group investors by the calendar year of their first transaction (2024 vs 2025) and compute their behavior.

In [ ]:
fact_transactions["transaction_date"] = pd.to_datetime(fact_transactions["transaction_date"])
first_txn = fact_transactions.groupby("investor_id")["transaction_date"].min().reset_index()
first_txn.columns = ["investor_id", "first_txn_date"]
first_txn["cohort_year"] = first_txn["first_txn_date"].dt.year

tx_cohorts = fact_transactions.merge(first_txn[["investor_id", "cohort_year"]], on="investor_id")

cohort_results = []
for year, group in tx_cohorts.groupby("cohort_year"):
    investor_count = group["investor_id"].nunique()
    sip_group = group[group["transaction_type"] == "SIP"]
    avg_sip = sip_group["amount_inr"].mean() if not sip_group.empty else 0.0
    total_invested = group[group["transaction_type"] != "Redemption"]["amount_inr"].sum()
    
    merged_with_fund = group.merge(dim_fund[["amfi_code", "category"]], on="amfi_code")
    pref_cat = merged_with_fund.groupby("category")["amount_inr"].sum().idxmax() if not merged_with_fund.empty else "None"
    
    cohort_results.append({
        "cohort_year": int(year),
        "total_investors": investor_count,
        "avg_sip_amount": avg_sip,
        "total_invested_amount": total_invested,
        "preferred_category": pref_cat
    })

cohort_df = pd.DataFrame(cohort_results)
print("Cohort Analysis Summary Table:")
print(cohort_df)

cohort_df.to_csv(REPORTS_DIR / "cohort_analysis.csv", index=False)

## 4. SIP Continuation Analysis & Churn Risk

For investors with 6 or more transactions, we track the gaps between consecutive SIPs. If the average gap is larger than 35 days, the investor is flagged as 'at-risk'.

In [ ]:
sip_tx = fact_transactions[fact_transactions["transaction_type"] == "SIP"].copy()
sip_tx = sip_tx.sort_values(["investor_id", "transaction_date"])

continuity_results = []
for investor_id, group in sip_tx.groupby("investor_id"):
    txn_count = len(group)
    if txn_count < 6:
        continue
    
    dates = group["transaction_date"].sort_values()
    gaps = dates.diff().dropna().dt.days
    avg_gap = gaps.mean()
    max_gap = gaps.max()
    at_risk = 1 if avg_gap > 35 else 0
    
    continuity_results.append({
        "investor_id": investor_id,
        "total_sip_transactions": txn_count,
        "avg_gap_days": avg_gap,
        "max_gap_days": max_gap,
        "at_risk_flag": at_risk
    })

continuity_df = pd.DataFrame(continuity_results)
total_investors_analyzed = len(continuity_df)
at_risk_count = continuity_df["at_risk_flag"].sum()
at_risk_pct = (at_risk_count / total_investors_analyzed) * 100 if total_investors_analyzed > 0 else 0.0

print(f"Total Investors Analyzed (6+ SIPs): {total_investors_analyzed}")
print(f"Investors Flagged as At-Risk: {at_risk_count} ({at_risk_pct:.2f}%)")
print("\nSample At-Risk Investors:")
print(continuity_df[continuity_df["at_risk_flag"] == 1].head())

continuity_df.to_csv(REPORTS_DIR / "sip_continuity.csv", index=False)

## 5. Sector Concentration Risk (HHI)

We calculate the Herfindahl-Hirschman Index (HHI) of sector weights for all equity portfolios to examine sector concentration risk.

In [ ]:
hhi_results = []
for code, group in fact_holdings.groupby("amfi_code"):
    scheme_name = dim_fund[dim_fund["amfi_code"] == code]["scheme_name"].values[0]
    cat = dim_fund[dim_fund["amfi_code"] == code]["category"].values[0]
    
    if cat != "Equity":
        continue
        
    sector_weights = group.groupby("sector")["weight_pct"].sum()
    
    if sector_weights.sum() > 0:
        norm_weights = sector_weights / sector_weights.sum() * 100
        hhi = np.sum(norm_weights ** 2)
    else:
        hhi = np.nan
        
    if hhi > 2500:
        level = "Highly Concentrated"
    elif hhi >= 1500:
        level = "Moderately Concentrated"
    else:
        level = "Diverse"
        
    hhi_results.append({
        "amfi_code": code,
        "scheme_name": scheme_name,
        "sector_hhi": hhi,
        "concentration_level": level
    })

hhi_df = pd.DataFrame(hhi_results).sort_values("sector_hhi", ascending=False).reset_index(drop=True)
print("Top 5 Equity Funds with Highest Sector Concentration:")
print(hhi_df.head())

hhi_df.to_csv(REPORTS_DIR / "sector_hhi.csv", index=False)

# Plot HHI Bar Chart
fig, ax = plt.subplots(figsize=(12, 6), dpi=150)
fig.patch.set_facecolor('#fafafa')
ax.set_facecolor('#ffffff')

plot_hhi = hhi_df.head(15)
ax.bar(plot_hhi["scheme_name"].apply(lambda x: x.split(" - ")[0]), plot_hhi["sector_hhi"], color=COLORS[0], width=0.5)
ax.axhline(2500, color=COLORS[3], linestyle="--", label="Highly Concentrated Threshold (>2500)")
ax.axhline(1500, color=COLORS[4], linestyle="--", label="Moderately Concentrated Threshold (1500-2500)")

ax.set_title("Sector Concentration Risk (Herfindahl-Hirschman Index - HHI)", fontsize=14, fontweight='bold', pad=15, color='#1e293b')
ax.set_xlabel("Scheme Name", fontsize=10, color='#475569', labelpad=10)
ax.set_ylabel("Sector HHI Score", fontsize=10, color='#475569', labelpad=10)
ax.tick_params(axis='x', rotation=30, labelsize=8)
ax.grid(True, linestyle="--", alpha=0.5, color="#cbd5e1")
ax.legend(loc="upper right", frameon=True, facecolor="white", edgecolor="#e2e8f0")

plt.savefig(REPORTS_DIR / "sector_hhi_chart.png", bbox_inches="tight", facecolor='#fafafa')
plt.show()

## 6. CLI Recommender Query Demo

We demonstrate how `recommender.py` queries the SQLite database to fetch top-performing funds by Sharpe ratio depending on the user's risk profile.

In [ ]:
# Demonstrate matching query for Low, Moderate, and High Appetites
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

risk_appetites = {
    "Low": ["Low"],
    "Moderate": ["Moderate", "Moderately High"],
    "High": ["High", "Very High"]
}

for appetite, grades in risk_appetites.items():
    placeholders = ",".join(["?"] * len(grades))
    query = f"""
        SELECT amfi_code, scheme_name, risk_grade, sharpe_ratio
        FROM fact_performance
        WHERE risk_grade IN ({placeholders})
        ORDER BY sharpe_ratio DESC
        LIMIT 3
    """
    cursor.execute(query, grades)
    rows = cursor.fetchall()
    
    print(f"=== Top 3 recommendations for {appetite} risk appetite ===")
    for row in rows:
        print(f"- {row[1]} ({row[2]} Risk, Sharpe: {row[3]:.4f})")
    print()

conn.close()

## Key Insights & Analytics Summary

1. **Downside Risk (VaR & CVaR)**:
   - Small-cap funds exhibit the highest VaR/CVaR, meaning they carry the largest potential daily loss (e.g. up to 1.8% daily in extreme market movements).
   - Liquid and Gilt funds exhibit extremely low VaR (below 0.1% daily), verifying their defensive role.
   
2. **Rolling Sharpe Ratio Stability**:
   - Major equity funds show highly volatile rolling Sharpe ratios, mirroring market trend shifts.
   - Tracking rolling risk-adjusted ratios helps investors identify when a fund is generating returns through excess volatility rather than alpha.
   
3. **Investor Cohorts (2024 vs 2025)**:
   - **2024 Cohort** contains the vast majority of investors (4,803 investors) with an average SIP of ~₹10,996.
   - **2025 Cohort** is smaller (197 investors) but has a higher average SIP (~₹13,505), indicating newer accounts are starting with larger commitment sizes.
   - Both cohorts prefer **Equity** as their main category choice.
   
4. **SIP Churn Risk**:
   - Among investors with 6+ SIP transactions, only **a small subset (12 investors, 0.88%)** are flagged as at-risk of churning based on average transaction gaps exceeding 35 days.
   - This low churn rate suggests high SIP continuation compliance across the customer base.
   
5. **Sector Concentration (HHI)**:
   - **Axis Bluechip Fund** has the highest sector concentration score (HHI = 2968), placing it in the **Highly Concentrated** risk profile (driven by heavy clustering in banking and finance).
   - Out of 35 analyzed equity funds, 4 are highly concentrated (HHI > 2500), 28 are moderately concentrated, and only 3 are highly diverse (HHI < 1500, e.g. Kotak Flexicap and UTI Mid Cap).